# Data Audit

Ce notebook relit les artefacts d'audit produits par `python -m src.project_runner`. Il sert de support public pour vérifier les fondations du projet : volume, cible, colonnes exclues, valeurs manquantes et modalités `Unknown`.

## Business Reading

Le churn est minoritaire. L'audit doit donc éviter deux pièges : traiter les churners comme une simple anomalie statistique, ou surestimer des signaux qui ne sont que des proxys comportementaux. Les colonnes `Naive_Bayes_Classifier_...` sont exclues immédiatement et `CLIENTNUM` reste un identifiant technique.

In [ ]:
from pathlib import Path
import json
import pandas as pd

root = Path('..').resolve()
audit_summary = json.loads((root / 'outputs' / 'metrics' / 'audit_summary.json').read_text())
missingness = pd.read_csv(root / 'outputs' / 'metrics' / 'missingness_profile.csv')
numeric_profile = pd.read_csv(root / 'outputs' / 'metrics' / 'numeric_profile.csv')
categorical_profile = pd.read_csv(root / 'outputs' / 'metrics' / 'categorical_profile.csv')
audit_summary

In [ ]:
missingness.sort_values(['missing_rate', 'unknown_count'], ascending=False).head(12)

## Unknown Values

Les modalités `Unknown` sont analysées comme une information de qualité de donnée et non comme un détail cosmétique. La V1 compare deux stratégies : conserver `Unknown` comme modalité ou le traiter comme missing avec indicateur dédié.

In [ ]:
categorical_profile[categorical_profile['value'].eq('Unknown')]

## Audit Takeaways

- Le dataset est suffisamment propre pour un cas portfolio, mais il reste statique et non causal.
- Le déséquilibre de classes impose une lecture PR-AUC / recall plutôt qu'une accuracy globale.
- Les valeurs `Unknown` doivent être documentées car elles peuvent encoder une absence d'information métier.